# Data Preparation Notebook
Run this notebook locally after downloading DIAGNOSES_ICD.csv.gz and NOTEEVENTS.csv.gz from the [MIMIC III dataset](https://physionet.org/content/mimiciii/1.4/).

In [2]:
# Imports and path specification
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

RANDOM_STATE = 42
TARGET_ADMISSIONS = 500

PROJECT_DIR = Path.home() / "Documents/Github/AI380H/MIMIC_NLP"

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

DIAGNOSES_FILE = RAW_DIR / "DIAGNOSES_ICD.csv.gz"
NOTES_FILE = RAW_DIR / "NOTEEVENTS.csv.gz"

OUTPUT_FILE = PROCESSED_DIR / "sepsis_discharge_notes.csv.gz"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Diagnoses:", DIAGNOSES_FILE)
print("Notes:", NOTES_FILE)
print("Output:", OUTPUT_FILE)

assert DIAGNOSES_FILE.exists(), "DIAGNOSES_ICD.csv.gz not found"
assert NOTES_FILE.exists(), "NOTEEVENTS.csv.gz not found"

Diagnoses: /Users/sghosh/Documents/Github/AI380H/MIMIC_NLP/data/raw/DIAGNOSES_ICD.csv.gz
Notes: /Users/sghosh/Documents/Github/AI380H/MIMIC_NLP/data/raw/NOTEEVENTS.csv.gz
Output: /Users/sghosh/Documents/Github/AI380H/MIMIC_NLP/data/processed/sepsis_discharge_notes.csv.gz


In [3]:
# Load the diagnoses table
diagnoses = pd.read_csv(
    DIAGNOSES_FILE,
    usecols=[
        "SUBJECT_ID",
        "HADM_ID",
        "SEQ_NUM",
        "ICD9_CODE"
    ],
    dtype={
        "SUBJECT_ID": "Int64",
        "HADM_ID": "Int64",
        "SEQ_NUM": "Int64",
        "ICD9_CODE": "string"
    }
)

diagnoses["ICD9_CODE"] = diagnoses["ICD9_CODE"].str.strip()

print("Diagnosis rows:", len(diagnoses))
print("Unique admissions:", diagnoses["HADM_ID"].nunique())

diagnoses.head()

Diagnosis rows: 651047
Unique admissions: 58976


,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE
0,109,172335,1,40301
1,109,172335,2,486
2,109,172335,3,58281
3,109,172335,4,5855
4,109,172335,5,4254


In [4]:
# Define the sepsis cohort
SEPSIS_EXACT_CODES = {
    "99591",   # Sepsis
    "99592",   # Severe sepsis
    "78552"    # Septic shock
}

sepsis_mask = (
    diagnoses["ICD9_CODE"].str.startswith("038", na=False)
    |
    diagnoses["ICD9_CODE"].isin(SEPSIS_EXACT_CODES)
)

sepsis_dx = diagnoses.loc[sepsis_mask].copy()

print("Sepsis diagnosis rows:", len(sepsis_dx))
print("Unique sepsis patients:", sepsis_dx["SUBJECT_ID"].nunique())
print("Unique sepsis admissions:", sepsis_dx["HADM_ID"].nunique())

sepsis_dx.head(10)

Sepsis diagnosis rows: 14182
Unique sepsis patients: 5593
Unique sepsis admissions: 6363


,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE
81,117,164853,10,0389
87,117,164853,16,99592
143,124,138376,5,0389
144,124,138376,6,99592
275,64,172056,2,03811
276,64,172056,3,99591
433,85,112077,1,0389
450,85,112077,18,99591
607,105,161160,2,0389
611,3,145834,1,0389


In [5]:
# Examine which codes appear
sepsis_code_counts = (
    sepsis_dx["ICD9_CODE"]
    .value_counts()
    .rename_axis("ICD9_CODE")
    .reset_index(name="COUNT")
)

sepsis_code_counts.head(20)

,ICD9_CODE,COUNT
0,99592,3912
1,0389,3725
2,78552,2586
3,99591,1272
4,03811,515
5,03842,467
6,03849,395
7,0380,376
8,0388,206
9,03819,150


In [6]:
# Create a reproducible candidate sample
sepsis_hadm_ids = (
    sepsis_dx["HADM_ID"]
    .dropna()
    .drop_duplicates()
    .astype(int)
)

# Oversample because some admissions may not have a usable
# discharge summary in NOTEEVENTS.
CANDIDATE_ADMISSIONS = min(
    max(TARGET_ADMISSIONS + 50, int(TARGET_ADMISSIONS * 1.25)),
    len(sepsis_hadm_ids)
)

candidate_hadm_ids = (
    sepsis_hadm_ids
    .sample(
        n=CANDIDATE_ADMISSIONS,
        random_state=RANDOM_STATE
    )
    .tolist()
)

candidate_hadm_set = set(candidate_hadm_ids)

print("Total sepsis admissions:", len(sepsis_hadm_ids))
print("Candidate admissions selected:", len(candidate_hadm_set))

Total sepsis admissions: 6363
Candidate admissions selected: 625


In [7]:
# Stream NOTEEVENTS.csv.gz instead of loading it
NOTE_COLUMNS = [
    "SUBJECT_ID",
    "HADM_ID",
    "CHARTDATE",
    "CATEGORY",
    "DESCRIPTION",
    "TEXT"
]

matching_chunks = []

CHUNK_SIZE = 20_000

reader = pd.read_csv(
    NOTES_FILE,
    usecols=NOTE_COLUMNS,
    dtype={
        "SUBJECT_ID": "Int64",
        "HADM_ID": "Int64",
        "CATEGORY": "string",
        "DESCRIPTION": "string"
    },
    chunksize=CHUNK_SIZE,
    low_memory=False
)

for chunk_number, chunk in enumerate(tqdm(reader, desc="Scanning NOTEEVENTS")):

    keep = (
        chunk["HADM_ID"].isin(candidate_hadm_set)
        &
        chunk["CATEGORY"].eq("Discharge summary")
        &
        chunk["TEXT"].notna()
    )

    if keep.any():
        matching_chunks.append(
            chunk.loc[keep].copy()
        )

print("Matching chunks found:", len(matching_chunks))

Scanning NOTEEVENTS: 105it [00:21,  4.99it/s]

Matching chunks found: 3


In [8]:
# Combine only the notes retained
if not matching_chunks:
    raise RuntimeError(
        "No matching discharge summaries were found."
    )

notes = pd.concat(
    matching_chunks,
    ignore_index=True
)

print("Discharge-summary rows extracted:", len(notes))
print("Admissions represented:", notes["HADM_ID"].nunique())

notes.head()

Discharge-summary rows extracted: 674
Admissions represented: 606


,SUBJECT_ID,HADM_ID,CHARTDATE,CATEGORY,DESCRIPTION,TEXT
0,80030,100442,2119-07-18,Discharge summary,Report,Admission Date: [**2119-6-7**] D...
1,24687,149107,2143-03-29,Discharge summary,Report,Admission Date: [**2143-3-10**] ...
2,21414,162875,2139-06-07,Discharge summary,Report,Admission Date: [**2139-6-3**] D...
3,3400,111651,2167-11-09,Discharge summary,Report,Admission Date: [**2167-11-5**] ...
4,9253,190600,2145-04-20,Discharge summary,Report,Admission Date: [**2145-4-13**] ...


In [9]:
# Clean obvious unusable records
notes["TEXT_LENGTH"] = (
    notes["TEXT"]
    .fillna("")
    .str.len()
)

notes["TEXT_LENGTH"].describe()

count      674.000000
mean     11802.939169
std       6526.458391
min        372.000000
25%       7616.000000
50%      10891.500000
75%      15398.500000
max      42099.000000
Name: TEXT_LENGTH, dtype: float64

In [10]:
# remove very short fragments
MIN_NOTE_LENGTH = 500

notes = notes.loc[
    notes["TEXT_LENGTH"] >= MIN_NOTE_LENGTH
].copy()

print("Notes after length filter:", len(notes))
print("Admissions remaining:", notes["HADM_ID"].nunique())

Notes after length filter: 672
Admissions remaining: 606


In [11]:
# Keep the longest discharge-summary document for each admission
notes = (
    notes
    .sort_values(
        ["HADM_ID", "TEXT_LENGTH"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset="HADM_ID",
        keep="first"
    )
    .reset_index(drop=True)
)

print("Unique discharge summaries:", len(notes))
print("Unique admissions:", notes["HADM_ID"].nunique())

Unique discharge summaries: 606
Unique admissions: 606


In [12]:
# Reduce to TARGET_ADMISSIONS amount of admissions
if len(notes) > TARGET_ADMISSIONS:
    notes = (
        notes
        .sample(
            n=TARGET_ADMISSIONS,
            random_state=RANDOM_STATE
        )
        .sort_values("HADM_ID")
        .reset_index(drop=True)
    )

print("Final number of discharge summaries:", len(notes))

Final number of discharge summaries: 500


In [13]:
# Add the qualifying sepsis ICD-9 codes to the notes
codes_per_admission = (
    sepsis_dx
    .groupby("HADM_ID")["ICD9_CODE"]
    .apply(
        lambda x: ",".join(
            sorted(set(x.dropna()))
        )
    )
    .rename("SEPSIS_ICD9_CODES")
)

notes = notes.merge(
    codes_per_admission,
    on="HADM_ID",
    how="left"
)

notes[
    [
        "SUBJECT_ID",
        "HADM_ID",
        "SEPSIS_ICD9_CODES",
        "TEXT_LENGTH"
    ]
].head(10)

,SUBJECT_ID,HADM_ID,SEPSIS_ICD9_CODES,TEXT_LENGTH
0,652,100262,"78552,99592",26850
1,54077,100454,"0389,99591",11396
2,7478,100539,03849,14651
3,23535,100598,"0389,99591",11038
4,29102,100995,"0389,99592",9949
5,28880,101090,"03849,78552,99592",13789
6,22660,101198,"0389,99591",5240
7,32286,101264,"0389,78552,99592",16312
8,70682,101441,"0388,99591",9548
9,22406,101904,"0389,78552,99592",10931


In [14]:
# Final validation
print("Rows:", len(notes))
print("Unique patients:", notes["SUBJECT_ID"].nunique())
print("Unique admissions:", notes["HADM_ID"].nunique())

print(
    "\nCategories:\n",
    notes["CATEGORY"].value_counts()
)

print(
    "\nMissing TEXT:",
    notes["TEXT"].isna().sum()
)

print(
    "\nDuplicate HADM_ID:",
    notes["HADM_ID"].duplicated().sum()
)

print(
    "\nNote-length summary:"
)

display(
    notes["TEXT_LENGTH"].describe()
)

# Some assertions
assert notes["TEXT"].notna().all()

assert (
    notes["CATEGORY"]
    .eq("Discharge summary")
    .all()
)

assert not notes["HADM_ID"].duplicated().any()

assert notes["SEPSIS_ICD9_CODES"].notna().all()

print("Validation passed.")


Rows: 500
Unique patients: 492
Unique admissions: 500

Categories:
 CATEGORY
Discharge summary    500
Name: count, dtype: Int64

Missing TEXT: 0

Duplicate HADM_ID: 0

Note-length summary:


count      500.000000
mean     12894.390000
std       6170.697126
min       1733.000000
25%       8743.500000
50%      11670.000000
75%      15803.750000
max      42099.000000
Name: TEXT_LENGTH, dtype: float64

Validation passed.


In [15]:
# Corpus statistics for the final dataset

total_characters = notes["TEXT"].str.len().sum()
mean_characters = notes["TEXT"].str.len().mean()

# Whitespace-delimited word count
total_words = notes["TEXT"].str.split().str.len().sum()
mean_words = notes["TEXT"].str.split().str.len().mean()

# Rough token estimate only
# Assumes approximately 5.5 characters per token
estimated_tokens = total_characters / 5.5

print(f"Total characters    : {total_characters:,}")
print(f"Mean chars/note     : {mean_characters:,.0f}")
print(f"Total words         : {total_words:,}")
print(f"Mean words/note     : {mean_words:,.0f}")
print(f"Estimated tokens    : {estimated_tokens:,.0f}")

Total characters    : 6,447,195
Mean chars/note     : 12,894
Total words         : 949,794
Mean words/note     : 1,900
Estimated tokens    : 1,172,217


In [16]:
# Save output to file
notes.to_csv(
    OUTPUT_FILE,
    index=False,
    compression="gzip"
)

print("Saved:", OUTPUT_FILE)
size_mb = (
    OUTPUT_FILE.stat().st_size
    / 1024**2
)

print(
    f"Compressed file size: {size_mb:.2f} MB"
)

Saved: /Users/sghosh/Documents/Github/AI380H/MIMIC_NLP/data/processed/sepsis_discharge_notes.csv.gz
Compressed file size: 2.24 MB


In [17]:
# Reload test
test_notes = pd.read_csv(
    OUTPUT_FILE,
    compression="gzip"
)

print(test_notes.shape)

display(
    test_notes[
        [
            "SUBJECT_ID",
            "HADM_ID",
            "CHARTDATE",
            "SEPSIS_ICD9_CODES",
            "TEXT_LENGTH"
        ]
    ].head()
)

(500, 8)


,SUBJECT_ID,HADM_ID,CHARTDATE,SEPSIS_ICD9_CODES,TEXT_LENGTH
0,652,100262,2142-06-11,"78552,99592",26850
1,54077,100454,2192-03-23,"0389,99591",11396
2,7478,100539,2106-04-02,03849,14651
3,23535,100598,2179-11-05,"0389,99591",11038
4,29102,100995,2135-03-15,"0389,99592",9949


In [18]:
assert len(test_notes) == len(notes)
assert test_notes["HADM_ID"].nunique() == len(test_notes)

print("Output file reload test passed.")

Output file reload test passed.
